# BLIP ITM-base COCO — DIMER image-text matching and retrieval tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/blip-itm-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/blip-itm-pipeline/blob/main/tutorials/blip_itm_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Salesforce%2Fblip--itm--base--coco-ffcc4d?style=flat)](https://huggingface.co/Salesforce/blip-itm-base-coco) [![Upstream](https://img.shields.io/badge/Upstream-salesforce%2FBLIP-181717?style=flat&logo=github&logoColor=white)](https://github.com/salesforce/BLIP) [![arXiv](https://img.shields.io/badge/arXiv-2201.12086-b31b1b.svg)](https://arxiv.org/abs/2201.12086)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** Image-text matching and retrieval — a grid of 1–16 images × 1–16 captions → an ITM match probability and an ITC cosine similarity per pair, and the captions ranked per image — using the pinned `Salesforce/blip-itm-base-coco` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/blip_itm_pipeline/pipeline.py` at revision `1310c594931e`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `bed8ad38cb2d04a5a4bdf2d071b3c3c0a4aa724c` (~896 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the BLIP retrieval model (a ViT-B/16 image encoder at 384×384, a BERT-style text encoder, and an image-grounded text encoder whose cross-attention fuses each caption with the image; about 224M parameters, pretrained on 129M image–text pairs with captioning-and-filtering bootstrapping and fine-tuned on COCO for retrieval) scores every image–caption pair twice: the **ITC head** compares the two projected embeddings by cosine similarity (the fast dual-encoder score used for retrieval), and the **ITM head** classifies the fused pair as match/no-match (the slower cross-attention score used for re-ranking); the carried module softmaxes the ITM logits into a match probability and ranks the captions per image by it. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (image side ceilings, 1–16 images, 1–16 distinct captions up to 256 characters), a fixed output contract (grids indexed `[image][text]`), and the `recall_at_1`, `validate_inputs` and `evaluation_report` helpers. **Weight-format note:** upstream hosts no SafeTensors at the pinned revision; the carried module executes the digest-pinned `pytorch_model.bin` (a pickle, deserialised with `weights_only=True` after its SHA-256 is checked), while the `tf_model.h5` upstream also hosts is DIMER's upload artifact and is never loaded here. The default sample is a 3×3 grid of cartoon scenes drawn in code and three authored captions, so recall@1 in both directions is demonstration (plumbing) evidence for one tiny grid, not a COCO retrieval benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision (a pickle checkpoint, and why that matters), draw three synthetic scenes and write their captions (or upload your own images and type captions) and validate them into an input manifest, run the supported task over the whole grid, read the two scores correctly (an uncalibrated match probability and a cosine similarity, neither an abstention), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with recall@1 in both directions only when the image–caption correspondence is known and `not-measurable` otherwise, and export the grids, a ranked contact sheet and provenance.

**This notebook does not demonstrate:** Captioning or question answering (separate checkpoints), retrieval over a corpus of thousands (the grid is capped at 16×16 and every pair is a full forward pass — an index of precomputed embeddings is a deployment's own build), recall@5/@10 or median rank (a 3×3 grid cannot express them), batch throughput, evaluation on COCO or Flickr30k (not bundled; only drawn scenes are scored here), and any training. The model was fine-tuned on COCO photographs with human captions; flat drawings, documents, non-English captions and captions that describe attributes the model ignores are outside what this notebook measures, and a high match probability carries no signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 3.9 s to load and 3.1 s for the nine pairs of the 3×3 grid in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 895 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python, NumPy and PIL; what a dual-encoder cosine score and a cross-attention match score are and why they differ; why a pickle checkpoint needs a digest check before `torch.load`; what recall@1 on a 3×3 grid does and does not show.
- **Data:** the default sample is three deterministic cartoon scenes drawn in code with Pillow (a house with a tree and the sun; a beach with a sailboat; two fruits on a table — no text rendering, so their digests are stable across Pillow builds) and three authored captions, one per scene, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one or more images decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px, plus your own captions typed into the form field; the correspondence is unknown for uploads, so their report is `not-measurable`. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Salesforce/blip-itm-base-coco` snapshot (~896 MB in total) at revision `bed8ad38cb2d…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'blip-itm-pipeline',
    'repository_revision': '1310c594931ec905ce60c079c60d5abacc26a54c',
    'embedded_module': 'src/blip_itm_pipeline/pipeline.py',
    'embedded_modules': ['src/blip_itm_pipeline/pipeline.py'],
    'module_sha256': 'eda58a70ae1f13116e4e0d6f154b8c8ca1880505b5ff3058cf9d628b2d0ef51c',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/blip_itm_pipeline/` @ `1310c594931e`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/blip_itm_pipeline/pipeline.py`

In [ ]:
"""Image-text matching and retrieval with the pinned ``Salesforce/blip-itm-base-coco`` checkpoint (BLIP).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the BLIP architecture comes from the pinned ``transformers`` release and no
model-repository code is executed. Upstream ships no SafeTensors at this revision: the PyTorch weights
are ``pytorch_model.bin`` (a pickle), so the trust boundary is the manifest SHA-256 checked before the
load plus ``weights_only=True`` deserialisation; the ``tf_model.h5`` upstream also hosts is the DIMER
upload artifact and is never loaded here. Two scores per image-caption pair: the ITM head's match
probability and the ITC cosine similarity.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "Salesforce/blip-itm-base-coco"
MODEL_REVISION = "bed8ad38cb2d04a5a4bdf2d071b3c3c0a4aa724c"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "blip-itm-base-coco"
WEIGHT_FILE = "pytorch_model.bin"  # the only PyTorch weight file upstream: a pickle, digest-pinned
HOSTED_TF_WEIGHT_FILE = "tf_model.h5"  # DIMER upload artifact (accepted format); never loaded by this package
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Grid ceilings. Every image-caption pair costs one fused forward pass (ITM) plus one dual-encoder pass
# (ITC), so the grid is bounded; a caption is one sentence for the BERT tokenizer.
MAX_IMAGES = 16
MAX_TEXTS = 16
MAX_TEXT_CHARS = 256
# Input ceilings. The processor resizes every image to 384x384 (preprocessor_config.json, aspect
# ratio not preserved) into 24x24 = 576 ViT-B/16 patches, so image cost is bounded; the side ceiling
# only guards memory during decoding and resizing.
IMAGE_SIZE = 384
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def format_texts(texts: Sequence[str]) -> list[str]:
    """Validate a list of captions: str, non-empty after whitespace collapse, within the ceiling, distinct."""
    if isinstance(texts, str) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a list of captions, not a single string")
    if not 1 <= len(texts) <= MAX_TEXTS:
        raise ValueError(f"caption count {len(texts)} outside 1..MAX_TEXTS {MAX_TEXTS}")
    cleaned: list[str] = []
    for text in texts:
        if not isinstance(text, str):
            raise TypeError(f"caption must be str, got {type(text).__name__}")
        collapsed = " ".join(text.split())
        if not collapsed:
            raise ValueError("captions must not be empty")
        if len(collapsed) > MAX_TEXT_CHARS:
            raise ValueError(
                f"caption {collapsed[:12]!r}... is {len(collapsed)} chars > MAX_TEXT_CHARS {MAX_TEXT_CHARS}"
            )
        cleaned.append(collapsed)
    if len(set(cleaned)) != len(cleaned):
        raise ValueError("captions must be distinct after whitespace normalisation")
    return cleaned


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def validate_images(images: Any) -> list[Image.Image]:
    if isinstance(images, Image.Image) or not isinstance(images, Sequence) or not images:
        raise TypeError("images must be a non-empty sequence of PIL.Image.Image")
    if len(images) > MAX_IMAGES:
        raise ValueError(f"image count {len(images)} > MAX_IMAGES {MAX_IMAGES}")
    return [validate_image(image) for image in images]


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "1..MAX_IMAGES PIL.Image.Image (any mode, converted to RGB) and 1..MAX_TEXTS caption strings; "
        "every image-caption pair is scored"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "images": [1, MAX_IMAGES],
    "texts": [1, MAX_TEXTS],
    "text_chars": [1, MAX_TEXT_CHARS],
    "preprocessing": (
        f"image resized to {IMAGE_SIZE}x{IMAGE_SIZE} (aspect ratio not preserved, CLIP mean/std) into 576 "
        "ViT-B/16 patches; caption tokenised by the snapshot's BERT tokenizer; per pair the ITC cosine "
        "similarity of the projected image and text embeddings and the ITM head's match/no-match logits "
        "over the fused representation"
    ),
    "output": (
        "per image-caption pair: itm_probability (softmax over the ITM head's two logits, a relative "
        "match score, not calibrated), the raw ITM logits and the ITC cosine similarity; per image the "
        "captions ranked by itm_probability"
    ),
}


def _check_inputs(images: Any, texts: Any) -> tuple[list[Image.Image], list[str]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``score`` and ``validate_inputs`` both route through this function so their acceptance criteria
    cannot diverge.
    """
    return validate_images(images), format_texts(texts)


def validate_inputs(
    images: Sequence[Image.Image],
    texts: Sequence[str],
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every image and caption is checked exactly as ``score`` would check it; rejection is reported by
    raising, and a caller that wants the finding recorded catches the exception and stores ``str(exc)``
    under ``findings``.
    """
    rgb, captions = _check_inputs(images, texts)
    if names is not None and len(names) != len(rgb):
        raise ValueError(f"names has {len(names)} entries for {len(rgb)} images")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[index] if names else f"image-{index}", "mode": image.mode, "size": list(image.size)}
            for index, image in enumerate(images)
        ],
        "texts": captions,
        "n_pairs": len(rgb) * len(captions),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def recall_at_1(scores: np.ndarray, correct: Sequence[int]) -> float:
    """Fraction of rows whose highest-scoring column is the labelled one (a square or rectangular grid)."""
    grid = np.asarray(scores, dtype=np.float64)
    if grid.ndim != 2 or grid.shape[0] != len(correct):
        raise ValueError(f"scores must be a 2-D grid with one correct column per row, got {grid.shape}")
    hits = 0
    for row, target in zip(grid, correct, strict=True):
        if isinstance(target, bool) or not isinstance(target, int) or not 0 <= target < grid.shape[1]:
            raise ValueError("each correct entry must be a valid zero-based column index")
        hits += int(np.argmax(row)) == target
    return hits / grid.shape[0]


def evaluation_report(
    result: Mapping[str, Any],
    correct_text_per_image: Sequence[int] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``correct_text_per_image`` (the zero-based index of each image's matching caption, in image
    order; a one-to-one grid is assumed for the text-to-image direction) the report carries image-to-text
    and text-to-image ``recall_at_1`` for both the ITM probability and the ITC cosine grid, the chance
    baseline, and the verdict ``sample-sanity``; without it the report is ``not-measurable`` and says what
    labelled data would make the task measurable.
    """
    itm = np.asarray(result["itm_probability"], dtype=np.float64)
    cosine = np.asarray(result["cosine"], dtype=np.float64)
    n_images, n_texts = itm.shape
    base = {
        "task": "image-text matching / retrieval over a caller-supplied grid of images and captions",
        "score_semantics": (
            "itm_probability is the softmax of the ITM head's match/no-match logits per pair (a relative "
            "match score, not calibrated, independent across pairs); cosine is the ITC similarity of the "
            "projected embeddings (comparable within a row or column, not a probability); no abstention"
        ),
        "sample_kind": sample_kind,
        "n_images": int(n_images),
        "n_texts": int(n_texts),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if correct_text_per_image is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no image-caption correspondence was supplied for the scored grid",
            "needs": (
                "captioned images from the deployment domain (COCO/Flickr30k-style, several captions per "
                "image) scored with recall@1/5/10 in both directions over thousands of candidates; no such "
                "labelled set ships with this repository"
            ),
        }
    if len(correct_text_per_image) != n_images:
        raise ValueError(
            f"correct_text_per_image has {len(correct_text_per_image)} entries for {n_images} images"
        )
    correct_image_per_text: list[int] | None = None
    if n_images == n_texts and sorted(correct_text_per_image) == list(range(n_texts)):
        inverse = {text: image for image, text in enumerate(correct_text_per_image)}
        correct_image_per_text = [inverse[text] for text in range(n_texts)]
    metrics = []
    for score_id, grid in (("itm_probability", itm), ("cosine", cosine)):
        metrics.append(
            {
                "id": f"image_to_text_recall_at_1_{score_id}",
                "value": recall_at_1(grid, correct_text_per_image),
                "estimation": f"{n_images} image(s) against {n_texts} caption(s), no dispersion estimate",
            }
        )
        if correct_image_per_text is not None:
            metrics.append(
                {
                    "id": f"text_to_image_recall_at_1_{score_id}",
                    "value": recall_at_1(grid.T, correct_image_per_text),
                    "estimation": f"{n_texts} caption(s) against {n_images} image(s), no dispersion estimate",
                }
            )
    return {
        **base,
        "metrics": metrics,
        "baselines": [
            {"id": "chance_image_to_text", "value": 1.0 / n_texts, "note": "random pick among the captions"},
            {"id": "chance_text_to_image", "value": 1.0 / n_images, "note": "random pick among the images"},
        ],
        "verdict": "sample-sanity",
        "reason": (
            f"a {n_images}x{n_texts} grid of images and captions you drew and wrote yourself; plumbing "
            "evidence, not a retrieval benchmark"
        ),
        "needs": (
            "a captioned image set from the deployment domain with thousands of candidates for any "
            "recall@k claim; COCO and Flickr30k are not bundled"
        ),
    }


@dataclass
class BlipItmPipeline:
    """``_runner(image, text)`` returns ``{"itm_logits": [no_match, match], "cosine": float}`` per pair."""

    _runner: Callable[[Image.Image, str], dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BlipItmPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import BlipForImageTextRetrieval, BlipProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = BlipProcessor.from_pretrained(location, **common)
        # Trust boundary: the only PyTorch weight file upstream is a pickle (pytorch_model.bin). Its
        # SHA-256 was checked against the manifest above; use_safetensors=False names that fact, and
        # weights_only=True makes transformers deserialise with torch.load(weights_only=True), whose
        # restricted unpickler admits tensors, primitives and containers only.
        model = BlipForImageTextRetrieval.from_pretrained(
            location, dtype=torch.float32, use_safetensors=False, weights_only=True, **common
        )
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, text: str) -> dict[str, Any]:
            inputs = processor(images=image, text=text, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                itm_logits = model(**inputs)[0]
                cosine = model(**inputs, use_itm_head=False)[0]
            return {
                "itm_logits": [float(v) for v in itm_logits[0].float().cpu().tolist()],
                "cosine": float(cosine.reshape(-1)[0]),
            }

        return cls(runner, resolved_device, "float32", source)

    def score(self, images: Sequence[Image.Image], texts: Sequence[str]) -> dict[str, Any]:
        """Score every image-caption pair; grids are indexed ``[image][text]``."""
        rgb, captions = _check_inputs(images, texts)
        itm_probability = np.zeros((len(rgb), len(captions)), dtype=np.float64)
        itm_logit_match = np.zeros_like(itm_probability)
        cosine = np.zeros_like(itm_probability)
        for i, image in enumerate(rgb):
            for j, caption in enumerate(captions):
                raw = self._runner(image, caption)
                if not isinstance(raw, dict) or "itm_logits" not in raw or "cosine" not in raw:
                    raise RuntimeError("runner must return a dict with 'itm_logits' and 'cosine'")
                logits = np.asarray(raw["itm_logits"], dtype=np.float64).reshape(-1)
                if logits.shape != (2,) or not np.all(np.isfinite(logits)):
                    raise RuntimeError(f"runner returned malformed ITM logits {raw['itm_logits']!r}")
                shifted = np.exp(logits - logits.max())
                itm_probability[i, j] = float(shifted[1] / shifted.sum())
                itm_logit_match[i, j] = float(logits[1])
                cosine[i, j] = float(raw["cosine"])
        rankings = [
            [
                {
                    "text": captions[j],
                    "itm_probability": float(itm_probability[i, j]),
                    "cosine": float(cosine[i, j]),
                }
                for j in np.argsort(-itm_probability[i])
            ]
            for i in range(len(rgb))
        ]
        return {
            "itm_probability": itm_probability,
            "itm_logit_match": itm_logit_match,
            "cosine": cosine,
            "rankings": rankings,
            "texts": captions,
            "n_images": len(rgb),
            "image_sizes": [list(image.size) for image in rgb],
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `bed8ad38cb2d…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BlipItmPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "blip-itm-base-coco",
  "modelId": "Salesforce/blip-itm-base-coco",
  "revision": "bed8ad38cb2d04a5a4bdf2d071b3c3c0a4aa724c",
  "files": [
    {
      "path": "README.md",
      "bytes": 5492,
      "sha256": "db2e7ff1e647bc42d8b0d4cd3653ad65c1d24b5559a153e6e6ea2f3c12edd978"
    },
    {
      "path": "config.json",
      "bytes": 4560,
      "sha256": "3e6464c2ce7c54512ddb101c5e9a8e77f4c2d637be9e3d005667ccd4a34c6ef2"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 445,
      "sha256": "0aa66e2e9ac3ea3b5cd4388c35072e22db4e1cc1f96c7872bed07749c712ade1"
    },
    {
      "path": "pytorch_model.bin",
      "bytes": 895139697,
      "sha256": "017fb3e7f4e125f13a8a4717f1402dbe0d0bb877474b4a203db13a4447b0227f"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 125,
      "sha256": "b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237ee3"
    },
    {
      "path": "tokenizer.json",
      "bytes": 711396,
      "sha256": "d241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 456,
      "sha256": "86da6fdb761b02f73a05561aba71711c2d7c205fe1fd1744046173a410263925"
    },
    {
      "path": "vocab.txt",
      "bytes": 231508,
      "sha256": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
    }
  ],
  "totalBytes": 896093679
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BlipItmPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic scenes or optional BYOD

The default sample is **synthetic** and carries its own references: three flat cartoon scenes — a red house with a tree, a white ball and the sun on grass under a blue sky; a beach with sea, sand, a red sailboat and the sun; a red apple and an orange on a wooden table — are drawn with Pillow (640×480, 640×480, 480×480), the same drawings the repository's smoke run used, and three captions are authored, one per scene, in that order. The diagonal of the grid is therefore the known correspondence and the reference for the recall@1 sanity check later. They are not a labelled dataset, so nothing here is a COCO retrieval measurement; the smoke run recorded a weak diagonal for the fruit scene (ITM 0.28 against its own caption) even though it still ranked first. The image digests are printed for the record. BYOD is optional and disabled by default; when enabled, upload one or more images and type your captions (one per line) — no correspondence is known for them, so the evaluation report will be `not-measurable`.

The caption set is a **caller-owned request parameter**: both scores rank only what you supply, and a caption set with nothing that fits an image still yields a highest-scoring caption. Nothing is validated in this cell — the next section hands the images and captions to the pipeline's own validation stage, which is the only checker. Look for one dictionary per image naming the sample kind, size and digest, plus the captions and the grid size.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
byod_captions = 'a dog on a beach\na plate of food'  # @param {type:"string"}


def synthetic_scenes():
    """Three flat cartoon scenes drawn with Pillow (no text); returns [(name, image, authored caption)]."""
    house = Image.new('RGB', (640, 480), (135, 206, 235))  # sky
    d = ImageDraw.Draw(house)
    d.rectangle([0, 300, 640, 480], fill=(60, 179, 75))  # grass
    d.ellipse([500, 40, 600, 140], fill=(255, 215, 0))  # sun
    d.rectangle([120, 180, 320, 330], fill=(200, 40, 40))  # red house
    d.polygon([(100, 180), (220, 90), (340, 180)], fill=(90, 50, 20))  # brown roof
    d.rectangle([200, 260, 240, 330], fill=(70, 40, 20))  # brown door
    d.ellipse([420, 260, 520, 360], fill=(40, 100, 40))  # tree crown
    d.rectangle([460, 350, 480, 420], fill=(90, 60, 30))  # trunk
    d.ellipse([60, 380, 140, 440], fill=(255, 255, 255))  # white ball
    beach = Image.new('RGB', (640, 480), (120, 190, 240))  # sky
    d = ImageDraw.Draw(beach)
    d.rectangle([0, 220, 640, 330], fill=(30, 110, 200))  # sea
    d.rectangle([0, 330, 640, 480], fill=(238, 214, 150))  # sand
    d.ellipse([60, 40, 150, 130], fill=(255, 230, 80))  # sun
    d.polygon([(400, 330), (470, 330), (435, 210)], fill=(230, 40, 40))  # red sail
    d.rectangle([432, 210, 438, 330], fill=(90, 60, 30))  # mast
    d.ellipse([200, 370, 260, 430], fill=(255, 120, 40))  # beach ball
    fruit = Image.new('RGB', (480, 480), (250, 250, 245))
    d = ImageDraw.Draw(fruit)
    d.ellipse([60, 120, 220, 280], fill=(220, 30, 30))  # red apple
    d.rectangle([135, 95, 145, 125], fill=(80, 50, 20))  # stalk
    d.ellipse([250, 140, 430, 300], fill=(255, 170, 20))  # orange
    d.polygon([(90, 400), (400, 400), (360, 330), (130, 330)], fill=(180, 120, 60))  # table
    return [
        ('synthetic_house_640x480.png', house, 'a red house with a tree under a blue sky'),
        ('synthetic_beach_640x480.png', beach, 'a sailboat on the sea next to a beach'),
        ('synthetic_fruit_480x480.png', fruit, 'an apple and an orange on a wooden table'),
    ]


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    names, images = [], []
    for name, data in uploaded.items():
        image = Image.open(io.BytesIO(data))
        image.load()
        names.append(name)
        images.append(image)
    texts = [line.strip() for line in byod_captions.splitlines() if line.strip()]
    correct_text_per_image = None
    sample_kind = 'BYOD'
else:
    # Deterministic drawings: no randomness and no text rendering, so no seed is needed and the digests are stable.
    samples = synthetic_scenes()
    names = [name for name, _, _ in samples]
    images = [image for _, image, _ in samples]
    texts = [caption for _, _, caption in samples]
    correct_text_per_image = list(range(len(samples)))  # caption i describes image i
    sample_kind = 'synthetic'

digests = {name: hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest() for name, image in zip(names, images)}
for name, image in zip(names, images):
    print({'sample_kind': sample_kind, 'name': name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': digests[name]})
print({'texts': texts, 'grid': [len(images), len(texts)], 'has_correspondence': correct_text_per_image is not None})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `score` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, at most `MAX_IMAGES` images, 1..`MAX_TEXTS` distinct non-empty captions of at most `MAX_TEXT_CHARS` characters (whitespace collapsed) — and returns an **input manifest** naming the schema (including the 384×384 resize that does not preserve aspect ratio and the two scores), each input's observed mode and size, the checked captions, the number of pairs and the verdict. The manifest is written to `outputs/blip_itm_input_manifest.json`. To show what rejection looks like, the cell also validates a duplicated caption and records the pipeline's own error message as a finding. Inside the pipeline each image is converted to RGB and resized to `IMAGE_SIZE`×`IMAGE_SIZE`; nothing else is dropped or altered. The pipeline cannot tell whether any caption describes any image: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'IMAGE_SIZE': IMAGE_SIZE, 'MAX_IMAGES': MAX_IMAGES, 'MAX_TEXTS': MAX_TEXTS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS}})
input_manifest = validate_inputs(images, texts, names=names)
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(images, [texts[0], '  ' + texts[0] + ' '])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'duplicate-caption-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/blip_itm_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Score the grid and read the output correctly

`score` returns three `[image][text]` grids — `itm_probability` (the softmax of the ITM head's match/no-match logits), `itm_logit_match` (the raw match logit) and `cosine` (the ITC similarity of the projected embeddings) — plus `rankings` (per image, the captions ordered by ITM probability), the checked captions, the image sizes and the model identity. **Neither score is calibrated and neither abstains**: the ITM probability is a per-pair classifier output that is not comparable to a human match rate, the cosine is comparable only within a row or a column, and a caption set with nothing that fits still produces a highest-scoring caption. Both are deterministic on a fixed device and dtype; CUDA kernels can shift them slightly, so GPU and CPU rankings need not agree on close pairs. Every pair costs one fused forward pass and one dual-encoder pass (about 0.35 s per pair on the reference CPU), so the grid scales as images × captions. As recorded in the model card, the repository's CPU smoke on this same grid put every image's own caption first by both scores — ITM 0.998 for the house, 0.634 for the beach and only 0.278 for the fruit scene — and gave a blank white image ITM 0.04–0.07 against all three captions: a low match probability is not evidence of an empty image, and a high one is not evidence of a correct description.

In [ ]:
import time

t0 = time.time()
result = pipe.score(images, texts)
elapsed = round(time.time() - t0, 2)
print({'device': pipe.device, 'dtype': pipe.dtype, 'seconds': elapsed, 'pairs': result['n_images'] * len(result['texts'])})
np.set_printoptions(precision=3, suppress=True)
print('itm_probability [image][text]:')
print(result['itm_probability'])
print('cosine [image][text]:')
print(result['cosine'])
for name, ranking in zip(names, result['rankings']):
    print(f"{name} -> " + '; '.join(f"{entry['text']!r} itm={entry['itm_probability']:.3f} cos={entry['cosine']:.3f}" for entry in ranking))

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No retrieval quality is reported by default: recall@k needs a captioned image set from the deployment domain with thousands of candidates, and this repository ships none (COCO and Flickr30k are not bundled). When the image–caption correspondence is supplied the report carries image-to-text and text-to-image `recall_at_1` for both the ITM probability and the cosine grid (the text-to-image direction only when the grid is square and one-to-one), the chance baselines, and the verdict `sample-sanity`. On the synthetic path the correspondence is the diagonal **you drew and wrote yourself**, so a perfect recall@1 on nine pairs proves only that the input contract, preprocessing, both heads and the ranking round-trip. On BYOD no correspondence is known, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/blip_itm_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, correct_text_per_image, sample_kind=sample_kind)
with open('outputs/blip_itm_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'baselines')}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:44} {metric['value']:.3f}  ({metric['estimation']})")
for baseline in report['baselines']:
    print(f"{baseline['id']:44} {baseline['value']:.3f}  ({baseline['note']})")
if report['verdict'] == 'not-measurable':
    print('No image-caption correspondence is known for this grid, so nothing is scored; read the rankings against the images yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the three grids, the rankings, the captions, the evaluation report, the input manifest, the sample identities, digests and correspondence, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the executed weight file and the hosted TensorFlow file it is not, and the runtime identity (Python, `torch`, `transformers`, device). The pair scores are also written as CSV with explicit `image`, `text`, `itm_probability`, `itm_logit_match`, `cosine` columns, and a contact-sheet PNG shows each image with its top-ranked caption and ITM probability for visual inspection — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
import csv

thumb_w, thumb_h, panel_h = 320, 240, 44
sheet = Image.new('RGB', (thumb_w * len(images), thumb_h + panel_h), 'white')
draw = ImageDraw.Draw(sheet)
panel_font = ImageFont.load_default(size=13)
for index, (image, ranking) in enumerate(zip(images, result['rankings'])):
    thumb = image.convert('RGB').copy()
    thumb.thumbnail((thumb_w, thumb_h))
    sheet.paste(thumb, (index * thumb_w + (thumb_w - thumb.width) // 2, (thumb_h - thumb.height) // 2))
    top = ranking[0]
    draw.text((index * thumb_w + 6, thumb_h + 6), f"itm {top['itm_probability']:.2f}: {top['text'][:44]}", fill=(40, 90, 220), font=panel_font)
sheet.save('outputs/blip_itm_annotated.png')
payload = {
    'itm_probability': result['itm_probability'].tolist(),
    'itm_logit_match': result['itm_logit_match'].tolist(),
    'cosine': result['cosine'].tolist(),
    'rankings': result['rankings'],
    'texts': result['texts'],
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'names': names, 'sizes': [list(image.size) for image in images], 'rgb_sha256': digests, 'correct_text_per_image': correct_text_per_image},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'executed_weight_file': WEIGHT_FILE,
    'hosted_tf_weight_file_not_loaded': HOSTED_TF_WEIGHT_FILE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/blip_itm_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/blip_itm_scores.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'text', 'itm_probability', 'itm_logit_match', 'cosine'])
    for i, name in enumerate(names):
        for j, text in enumerate(result['texts']):
            writer.writerow([name, text, f"{result['itm_probability'][i, j]:.6f}", f"{result['itm_logit_match'][i, j]:.4f}", f"{result['cosine'][i, j]:.6f}"])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The scores are a match classifier's probability and an embedding cosine for pairs you supplied; nothing in the output says whether any caption truly describes any image, neither score is calibrated, and the model ranks every grid — including a blank image — without abstaining. On the drawn scenes the recall@1 values in the evaluation report compare the rankings with a correspondence you drew and wrote yourself and the verdict is `sample-sanity`, which proves only that the input contract, preprocessing, both heads and the ranking work (the repository's smoke run scored 1.0 in both directions by both scores on nine pairs, with the fruit scene's own caption winning at only 0.28); it says nothing about photographs, near-duplicate captions, attribute-level distinctions (`a red house` 0.64 vs `a blue house` 0.005 in the smoke run, but `a house` 0.30), non-English captions, or retrieval over thousands of candidates, and a BYOD result is a single-grid observation with the verdict `not-measurable`. **The caption set is part of the request**: the highest-scoring caption for an image is the best of what you offered, not a description. The pipeline provides no captioning, no corpus index, no abstention, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model (a pickle checkpoint loaded with `weights_only=True` only after its digest matched), validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** add a fourth caption that fits none of the scenes and watch which image claims it; replace the fruit caption with `an apple and a banana` and compare the ITM probability with the cosine; enable `USE_BYOD` with photographs you know, type their captions, then pass the true correspondence to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/blip-itm-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/blip-itm-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/blip-itm-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Salesforce/blip-itm-base-coco
- Upstream code: https://github.com/salesforce/BLIP
- BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation (Li et al., 2022): https://arxiv.org/abs/2201.12086
- Microsoft COCO Captions: Data Collection and Evaluation Server (Chen et al., 2015): https://arxiv.org/abs/1504.00325